# GRADE --- RoBERTa on the two extreme folds

The main sweep (`GRADE_full_study_v2.ipynb`) trains DeBERTa-v3-small on
all 15 leave-one-assignment-out folds. The paper's "One architecture"
limitation says the result --- accuracy survives the shift but the
word-frequency control matches or beats the detector on 9 of 15 folds ---
describes `deberta-v3-small` and not encoders in general.

This notebook is the cheapest check of that: it trains **RoBERTa-base**
on the same data, same splits, same recipe, on only the two folds that
matter for the question "is this architecture-specific" --- the best and
worst DeBERTa folds:

| Fold | Held-out assignment | DeBERTa accuracy |
|---|---|---|
| 13 | Exploring Venus | 93.47% (worst) |
| 1  | Car-free cities | 99.61% (best) |

It writes into the same `results/loto_runs.csv`, `results/loto_baselines.csv`
and `results/loto_predictions/` that the main sweep uses, tagged
`model='roberta'`, so the two are directly comparable without duplicating
any data-prep or splitting logic. It does not touch, re-run, or depend on
anything in the main notebook beyond the fold CSVs already on disk in
`data/loto/`.

Runs both folds at seed 42 first (to compare directly against the
DeBERTa row at seed 42), then repeats them at seeds 123 and 2024 if
`STAGE = 2`, mirroring the seed-repetition rule already used for DeBERTa's
extremes.

## 0. Environment

In [1]:
import os, sys, gc, time, random, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')

import numpy as np
import pandas as pd

REPO    = os.path.abspath('..')
DATA    = os.path.join(REPO, 'data')
LOTO    = os.path.join(DATA, 'loto')
RESULTS = os.path.join(REPO, 'results')
PREDS   = os.path.join(RESULTS, 'loto_predictions')
CKPT    = os.path.join(REPO, 'models', 'loto')
for d in (PREDS, CKPT):
    os.makedirs(d, exist_ok=True)

RUNS = os.path.join(RESULTS, 'loto_runs.csv')
BASE = os.path.join(RESULTS, 'loto_baselines.csv')

import torch
print('torch          ', torch.__version__)
print('cuda available ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device         ', torch.cuda.get_device_name(0))
    print('vram           ', '%.1f GB' %
          (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('\n*** NO GPU - this will be impractically slow. ***')
import transformers
print('transformers   ', transformers.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

torch           2.5.1+cu118
cuda available  True
device          NVIDIA GeForce RTX 3060 Ti
vram            8.6 GB
transformers    4.57.6


## 1. Configuration

In [2]:
STAGE = 1                     # 1 = seed 42 on both folds; 2 = extra seeds too
MODEL = 'roberta'
HF_ID = 'roberta-base'
EXTRA_SEEDS = [123, 2024]

# The two DeBERTa extremes from the completed 15-fold sweep (seed 42):
#   fold 13, Exploring Venus      -- worst,  93.47% accuracy
#   fold  1, Car-free cities      -- best,   99.61% accuracy
EXTREME_FOLDS = [13, 1]

CONFIG = {'max_len': 256, 'lr': 2e-5, 'batch_size': 8, 'epochs': 3}

print('detector %s -> %s' % (MODEL, HF_ID))
print('folds    %s' % EXTREME_FOLDS)
print('config   %s' % CONFIG)
print('stage    %d' % STAGE)

detector roberta -> roberta-base
folds    [13, 1]
config   {'max_len': 256, 'lr': 2e-05, 'batch_size': 8, 'epochs': 3}
stage    1


## 2. Training and evaluation core

Identical to the main notebook's section 4 core: same dataset class, same
encode/predict/metrics functions, same AdamW recipe, same best-checkpoint
selection on validation accuracy. Reproduced here rather than imported so
this notebook runs standalone.

In [3]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, recall_score, roc_auc_score,
                             precision_score, f1_score, confusion_matrix)
from tqdm.auto import tqdm

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class TextDS(Dataset):
    def __init__(self, df):
        self.t = df['text'].astype(str).tolist()
        self.y = df['label'].astype(int).tolist()
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i], self.y[i]

def loader(df, bs, shuffle):
    return DataLoader(TextDS(df), batch_size=bs, shuffle=shuffle)

def encode(tok, texts, ml):
    return tok(list(texts), max_length=ml, truncation=True,
               padding='max_length', return_tensors='pt').to(DEVICE)

@torch.no_grad()
def predict(model, tok, df, cfg, desc='eval'):
    model.eval(); ys, ps, prs = [], [], []
    for texts, y in tqdm(loader(df, cfg['batch_size'], False), desc=desc, leave=False):
        lg = model(**encode(tok, texts, cfg['max_len'])).logits
        prs.extend(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        ps.extend(torch.argmax(lg, 1).cpu().numpy()); ys.extend(np.asarray(y))
    return np.array(ys), np.array(ps), np.array(prs)

def metrics(y, p, pr):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y, p),
            'precision': precision_score(y, p, zero_division=0),
            'recall': recall_score(y, p, zero_division=0),
            'f1': f1_score(y, p, zero_division=0),
            'roc_auc': roc_auc_score(y, pr) if len(set(y)) > 1 else float('nan'),
            'fpr': fp / (fp + tn) if (fp + tn) else float('nan'),
            'fnr': fn / (fn + tp) if (fn + tp) else float('nan'),
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def train_one(hf_id, seed, cfg, tr, va, tag):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'])
    tl = loader(tr, cfg['batch_size'], True)
    best, state = -1.0, None
    for ep in range(cfg['epochs']):
        model.train(); t0, tot = time.time(), 0.0
        for texts, y in tqdm(tl, desc='%s ep%d/%d' % (tag, ep+1, cfg['epochs']),
                             leave=False):
            yt = torch.as_tensor(y).to(DEVICE); opt.zero_grad()
            out = model(**encode(tok, texts, cfg['max_len']), labels=yt)
            out.loss.backward(); opt.step(); tot += out.loss.item()
        yv, pv, _ = predict(model, tok, va, cfg, desc='val')
        vacc = float((yv == pv).mean())
        print('   ep%d loss=%.4f val_acc=%.4f (%.0fs)'
              % (ep+1, tot/max(len(tl), 1), vacc, time.time()-t0), flush=True)
        if vacc > best:
            best = vacc
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if state is not None:
        model.load_state_dict(state); model.to(DEVICE)
    return model, tok, best

def done_keys():
    if not os.path.exists(RUNS):
        return set()
    d = pd.read_csv(RUNS)
    return set(zip(d['model'], d['fold'], d['seed']))

print('core defined')

core defined


## 3. Controls (re-fit here for reference, not written to results)

The length-only and word-frequency controls are already fitted per fold
in the main sweep and stored in `results/loto_baselines.csv` --- they do
not depend on the detector architecture, only on the fold's train/test
split, which is identical here. This cell re-derives them for the two
extreme folds purely to display alongside RoBERTa's numbers in section 5;
it does **not** append to `loto_baselines.csv`, since that file already
has these rows from the DeBERTa run and re-adding them would duplicate.

In [4]:
base = pd.read_csv(BASE) if os.path.exists(BASE) else pd.DataFrame()
if not base.empty:
    show = base[base.fold.isin(EXTREME_FOLDS)].pivot_table(
        index='fold', columns='baseline', values='accuracy')
    print(show.round(4))
else:
    print('results/loto_baselines.csv not found -- run the main sweep first, '
          'or continue: this notebook does not need it to train.')

baseline  length_only   tfidf
fold                         
1              0.5497  0.9932
13             0.5287  0.9873


## 4. Train

Resumable: re-running this cell skips any (model, fold, seed) already in
`results/loto_runs.csv`. Writes predictions to
`results/loto_predictions/preds_f{fold:02d}_roberta_seed{seed}.csv` in the
same format the main sweep uses.

In [5]:
if STAGE == 1:
    jobs = [(f, 42) for f in EXTREME_FOLDS]
else:
    jobs = [(f, s) for f in EXTREME_FOLDS for s in EXTRA_SEEDS]
print('%d run(s) planned: %s' % (len(jobs), jobs))

topics = {13: 'Exploring Venus', 1: 'Car-free cities'}

for fold, seed in jobs:
    if (MODEL, fold, seed) in done_keys():
        print('[skip] fold %02d seed %d done' % (fold, seed)); continue
    topic = topics.get(fold, '?')
    tr = pd.read_csv(os.path.join(LOTO, 'fold_%02d_train.csv' % fold))
    va = pd.read_csv(os.path.join(LOTO, 'fold_%02d_val.csv' % fold))
    te = pd.read_csv(os.path.join(LOTO, 'fold_%02d_test.csv' % fold))
    print('=== fold %02d seed %d  held out: %s ===' % (fold, seed, topic))
    print('    train=%d val=%d test=%d' % (len(tr), len(va), len(te)), flush=True)
    try:
        model, tok, best_val = train_one(HF_ID, seed, CONFIG, tr, va,
                                         'f%02d s%d' % (fold, seed))
    except Exception as ex:
        print('[FAIL] %s: %s' % (type(ex).__name__, ex)); continue

    y, p, pr = predict(model, tok, te, CONFIG, desc='test')
    m = metrics(y, p, pr)
    pd.DataFrame({'uid': te['uid'].values, 'label': y, 'pred': p,
                  'prob_ai': pr}).to_csv(
        os.path.join(PREDS, 'preds_f%02d_%s_seed%d.csv' % (fold, MODEL, seed)),
        index=False)
    pd.DataFrame([{'model': MODEL, 'hf_id': HF_ID, 'fold': fold,
                   'held_out_topic': topic, 'seed': seed, 'n_train': len(tr),
                   'n_val': len(va), 'n_test': len(te),
                   'best_val_accuracy': best_val, **CONFIG, **m}]).to_csv(
        RUNS, mode='a', header=not os.path.exists(RUNS), index=False)
    print('    acc=%.4f recall=%.4f roc=%.4f fpr=%.4f fnr=%.4f'
          % (m['accuracy'], m['recall'], m['roc_auc'], m['fpr'], m['fnr']),
          flush=True)

    del model, tok; gc.collect(); torch.cuda.empty_cache()

print('\nSTAGE %d COMPLETE' % STAGE)

2 run(s) planned: [(13, 42), (1, 42)]
=== fold 13 seed 42  held out: Exploring Venus ===
    train=29206 val=5154 test=628


f13 s42 ep1/3:   0%|          | 0/3651 [00:00<?, ?it/s]

val:   0%|          | 0/645 [00:00<?, ?it/s]

   ep1 loss=0.0429 val_acc=0.9955 (734s)


f13 s42 ep2/3:   0%|          | 0/3651 [00:00<?, ?it/s]

...

f01 s42 ep3/3:   0%|          | 0/3282 [00:00<?, ?it/s]

val:   0%|          | 0/579 [00:00<?, ?it/s]

   ep3 loss=0.0099 val_acc=0.9996 (622s)


test:   0%|          | 0/513 [00:00<?, ?it/s]

    acc=0.9959 recall=0.9937 roc=0.9999 fpr=0.0020 fnr=0.0063

STAGE 1 COMPLETE


## 5. Compare against DeBERTa on the same two folds

Pulls both models' rows for folds 13 and 1 out of the shared
`loto_runs.csv`, alongside the TF-IDF control, so the comparison the
paper's "One architecture" limitation calls for is one table away.

In [6]:
runs = pd.read_csv(RUNS)
base = pd.read_csv(BASE) if os.path.exists(BASE) else pd.DataFrame()

sub = runs[(runs.fold.isin(EXTREME_FOLDS)) & (runs.seed == 42)]
piv = base[base.fold.isin(EXTREME_FOLDS)].pivot_table(
    index='fold', columns='baseline', values='accuracy') if not base.empty else None

out = sub[['model', 'fold', 'held_out_topic', 'accuracy', 'recall',
          'roc_auc', 'fpr']].sort_values(['fold', 'model'])
if piv is not None:
    out = out.merge(piv, on='fold', how='left')
for c in ['accuracy', 'recall', 'roc_auc', 'fpr'] + (
        list(piv.columns) if piv is not None else []):
    if c in out:
        out[c] = (out[c] * 100).round(2)
print(out.to_string(index=False))

     model  fold  held_out_topic  accuracy  recall  roc_auc   fpr  length_only  tfidf
deberta-v3     1 Car-free cities     99.61   99.85    99.99  0.63        54.97  99.32
   roberta     1 Car-free cities     99.59   99.37    99.99  0.20        54.97  99.32
deberta-v3    13 Exploring Venus     93.47  100.00    99.98 13.06        52.87  98.73
   roberta    13 Exploring Venus     99.84   99.68   100.00  0.00        52.87  98.73
